# 01 - Filtros Classicos: Hodrick-Prescott, Baxter-King e Christiano-Fitzgerald

Neste notebook, exploramos os tres filtros mais utilizados na macroeconomia para decompor
series temporais em componentes de **tendencia** e **ciclo**:

1. **Hodrick-Prescott (HP)** - o mais popular, porem controverso
2. **Baxter-King (BK)** - filtro band-pass simetrico
3. **Christiano-Fitzgerald (CF)** - band-pass assimetrico (full-sample)

### Conteudo
- Teoria e intuicao de cada filtro
- Implementacao com chronobox
- Discussao sobre a escolha de $\lambda$ no HP
- Criticas ao filtro HP (Hamilton 2018, Ravn-Uhlig)
- Comparacao visual dos tres filtros
- Exercicios praticos

### Referencias
- Hodrick, R.J. & Prescott, E.C. (1997). *Postwar U.S. Business Cycles: An Empirical Investigation*. Journal of Money, Credit, and Banking.
- Baxter, M. & King, R.G. (1999). *Measuring Business Cycles: Approximate Band-Pass Filters for Economic Time Series*. Review of Economics and Statistics.
- Christiano, L.J. & Fitzgerald, T.J. (2003). *The Band Pass Filter*. International Economic Review.
- Ravn, M.O. & Uhlig, H. (2002). *On Adjusting the Hodrick-Prescott Filter for the Frequency of Observations*. Review of Economics and Statistics.
- Hamilton, J.D. (2018). *Why You Should Never Use the Hodrick-Prescott Filter*. Review of Economics and Statistics.

## 1. Setup e Carregamento dos Dados

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os

# Adicionar o diretorio raiz do projeto ao path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Imports do chronobox
from chronobox.filters import hp_filter, bk_filter, cf_filter

# Helpers
sys.path.insert(0, os.path.join(project_root, 'examples', 'filters'))
from utils.plot_helpers import plot_trend_cycle, plot_bandpass_comparison
from utils.data_generators import generate_trend_cycle

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100

print('Setup completo!')

In [ ]:
# Carregar PIB real dos EUA (trimestral, 1970-)
data_dir = os.path.join(project_root, 'examples', 'filters', 'data')
us_gdp = pd.read_csv(os.path.join(data_dir, 'us_gdp_quarterly.csv'), parse_dates=['date'])

print(f'Periodo: {us_gdp["date"].iloc[0].date()} a {us_gdp["date"].iloc[-1].date()}')
print(f'Observacoes: {len(us_gdp)}')
print(f'\nPrimeiras linhas:')
us_gdp.head()

## 2. Filtro Hodrick-Prescott (HP)

O filtro HP decompoe a serie $y_t$ em tendencia $\tau_t$ e ciclo $c_t = y_t - \tau_t$,
resolvendo o problema de minimizacao:

$$\min_{\tau_t} \left\{ \sum_{t=1}^{T} (y_t - \tau_t)^2 + \lambda \sum_{t=2}^{T-1} [(\tau_{t+1} - \tau_t) - (\tau_t - \tau_{t-1})]^2 \right\}$$

O parametro $\lambda$ controla a suavidade da tendencia:
- $\lambda \to 0$: tendencia = serie original (sem suavizacao)
- $\lambda \to \infty$: tendencia linear

### Valores convencionais de $\lambda$

| Frequencia | $\lambda$ | Referencia |
|:-----------|----------:|:----------|
| Anual      | 6.25      | Ravn & Uhlig (2002) |
| Trimestral | 1,600     | Hodrick & Prescott (1997) |
| Mensal     | 129,600   | Ravn & Uhlig (2002) |

A regra de Ravn-Uhlig sugere escalar $\lambda$ pela quarta potencia da razao de frequencias:
$\lambda_f = \lambda_{\text{trim}} \times \left(\frac{f}{4}\right)^4$

In [ ]:
# HP filter com lambda=1600 (padrao trimestral)
y = us_gdp['gdp_log'].values
dates = us_gdp['date']

hp_trend, hp_cycle = hp_filter(y, lamb=1600)

print(f'HP Filter (lambda=1600)')
print(f'  Desvio padrao do ciclo: {hp_cycle.std():.4f}')
print(f'  Min ciclo: {hp_cycle.min():.4f}')
print(f'  Max ciclo: {hp_cycle.max():.4f}')

In [ ]:
# Grafico 1: Decomposicao HP do PIB dos EUA
fig = plot_trend_cycle(
    dates, y, hp_trend, hp_cycle,
    title='Filtro HP (λ=1600) - PIB Real EUA (log)'
)
plt.show()

### 2.1 Sensibilidade ao parametro $\lambda$

Vamos ver como diferentes valores de $\lambda$ afetam a decomposicao:

In [ ]:
# Grafico 2: Comparando diferentes lambdas
lambdas = [100, 1600, 10000, 129600]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for i, lam in enumerate(lambdas):
    trend_i, cycle_i = hp_filter(y, lamb=lam)
    
    axes[i].plot(dates, y, alpha=0.5, label='Observado', linewidth=0.8)
    axes[i].plot(dates, trend_i, label=f'Tendencia (λ={lam:,})', linewidth=2)
    axes[i].set_title(f'λ = {lam:,}')
    axes[i].legend(fontsize=9)
    axes[i].grid(True, alpha=0.3)

fig.suptitle('Sensibilidade do Filtro HP ao Parametro λ', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('Desvio padrao do ciclo por lambda:')
for lam in lambdas:
    _, c = hp_filter(y, lamb=lam)
    print(f'  λ={lam:>7,}: std(ciclo) = {c.std():.4f}')

In [ ]:
# Tambem podemos usar o parametro frequency para escolha automatica
hp_trend_auto, hp_cycle_auto = hp_filter(y, frequency='quarterly')

# Verificar que frequency='quarterly' equivale a lamb=1600
print(f'Diferenca maxima entre lamb=1600 e frequency="quarterly": {np.max(np.abs(hp_trend - hp_trend_auto)):.2e}')

### 2.2 Criticas ao Filtro HP

Apesar de ser o filtro mais popular em macroeconomia, o HP tem limitacoes serias:

**Hamilton (2018) - "Why You Should Never Use the HP Filter":**
1. **Ciclos espurios**: o HP pode gerar dinamicas ciclicas que nao existem nos dados originais
2. **Problemas de endpoint**: a tendencia estimada e fortemente influenciada pelas ultimas observacoes
3. **Nao-fundamentado**: nao ha justificativa teorica para o valor $\lambda = 1600$
4. **Variaveis integradas**: aplicar HP a processos com raiz unitaria produz resultados espurios

**Ravn & Uhlig (2002):**
- Propoe a regra $\lambda_f = 1600 \times (f/4)^4$ para ajustar a frequencia
- Mostra que sem ajuste, comparacoes entre frequencias sao invalidas

Nas proximas secoes, veremos alternativas mais robustas: BK, CF e, no proximo notebook, o filtro de Hamilton.

## 3. Filtro Baxter-King (BK)

O filtro BK e um **band-pass filter** que isola flutuacoes dentro de uma banda de frequencias especifica.
Para ciclos de negocios, a banda tipica e de **6 a 32 trimestres** (1.5 a 8 anos).

O filtro e construido como uma media movel simetrica com pesos $a_j$:

$$c_t = \sum_{j=-K}^{K} a_j \, y_{t-j}$$

onde $K$ (truncation) controla o numero de leads/lags. O BK perde $K$ observacoes em cada
extremo da amostra.

**Vantagens do BK:**
- Fundamentacao espectral solida (band-pass ideal aproximado)
- Sem problemas de endpoint (regioes com dados insuficientes sao descartadas)

**Desvantagens:**
- Perda de observacoes nas bordas ($2K$ no total)
- Simetrico: nao se adapta ao tamanho da amostra

In [ ]:
# BK filter: ciclos entre 6 e 32 trimestres, truncamento K=12
bk_cycle = bk_filter(y, low=6, high=32, trunc=12)

K = 12
print(f'BK Filter (low=6, high=32, trunc={K})')
print(f'  Obs originais: {len(y)}')
print(f'  Obs apos filtro: {len(bk_cycle)} (perdeu {2*K} nas bordas)')
print(f'  Desvio padrao do ciclo: {bk_cycle.std():.4f}')

In [ ]:
# Grafico 3: Ciclo BK
bk_dates = dates.iloc[K:-K]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(bk_dates, bk_cycle, label='Ciclo BK (6-32 trimestres)', color='tab:red', linewidth=1.2)
ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.fill_between(bk_dates, bk_cycle, 0, alpha=0.2, color='tab:red')
ax.set_title('Filtro Baxter-King - Componente Ciclico do PIB EUA (log)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Filtro Christiano-Fitzgerald (CF)

O filtro CF e outra aproximacao ao band-pass ideal, com duas vantagens importantes:

1. **Full-sample**: usa todas as observacoes (nao perde dados nas bordas)
2. **Assimetrico**: adapta os pesos nos endpoints da amostra

A ideia e usar a representacao no dominio da frequencia para construir um filtro otimo
que minimiza o erro quadratico medio em relacao ao filtro band-pass ideal.

O CF assume que a serie original segue um passeio aleatorio, e a partir disso
deriva pesos assimetricos para as primeiras e ultimas observacoes.

In [ ]:
# CF filter: mesma banda de 6-32 trimestres
cf_cycle = cf_filter(y, low=6, high=32, drift=False)

print(f'CF Filter (low=6, high=32)')
print(f'  Obs originais: {len(y)}')
print(f'  Obs apos filtro: {len(cf_cycle)} (sem perda!)')
print(f'  Desvio padrao do ciclo: {cf_cycle.std():.4f}')

In [ ]:
# Grafico 4: Ciclo CF
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(dates, cf_cycle, label='Ciclo CF (6-32 trimestres)', color='tab:green', linewidth=1.2)
ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.fill_between(dates, cf_cycle, 0, alpha=0.2, color='tab:green')
ax.set_title('Filtro Christiano-Fitzgerald - Componente Ciclico do PIB EUA (log)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Comparacao Visual: HP vs BK vs CF

Vamos sobrepor os componentes ciclicos estimados pelos tres filtros para identificar
diferencas e semelhancas. Para comparacao justa, usamos apenas o intervalo onde
o BK tem dados (excluindo as bordas).

In [ ]:
# Preparar ciclos para comparacao (alinhados ao intervalo do BK)
K = 12

# HP cycle ja tem tamanho T, recortar para o intervalo do BK
hp_cycle_trim = hp_cycle[K:-K]
# CF cycle tambem tem tamanho T
cf_cycle_trim = cf_cycle[K:-K]
# BK cycle ja tem tamanho T - 2K
bk_dates = dates.iloc[K:-K]

# Grafico 5: Comparacao dos tres filtros
cycles = {
    'HP (λ=1600)': hp_cycle_trim,
    'BK (6-32, K=12)': bk_cycle,
    'CF (6-32)': cf_cycle_trim
}

fig = plot_bandpass_comparison(
    bk_dates, cycles,
    title='Comparacao de Filtros: Ciclo do PIB Real EUA (log)'
)
plt.show()

# Correlacoes entre os ciclos
print('Correlacoes entre componentes ciclicos:')
print(f'  HP vs BK: {np.corrcoef(hp_cycle_trim, bk_cycle)[0,1]:.4f}')
print(f'  HP vs CF: {np.corrcoef(hp_cycle_trim, cf_cycle_trim)[0,1]:.4f}')
print(f'  BK vs CF: {np.corrcoef(bk_cycle, cf_cycle_trim)[0,1]:.4f}')

### 5.1 Resumo Comparativo

| Filtro | Tipo | Perda de obs | Lambda/$K$ | Endpoints |
|:-------|:-----|:-------------|:-----------|:----------|
| HP     | Low-pass/High-pass | Nenhuma | $\lambda=1600$ (trim.) | Problematicos |
| BK     | Band-pass simetrico | $2K$ | $K=12$ (trim.) | Removidos |
| CF     | Band-pass assimetrico | Nenhuma | - | Ajuste assimetrico |

**Observacoes:**
- BK e CF extraem **ciclos na mesma banda** (6-32 trim) e tendem a concordar no meio da amostra
- HP nao e estritamente band-pass: passa frequencias fora da banda de ciclos de negocios
- BK e mais conservador (descarta bordas); CF e mais agressivo nos endpoints

## 6. Dados Sinteticos: Validacao com Ciclo Conhecido

Vamos testar os filtros em dados sinteticos onde sabemos o ciclo verdadeiro:

In [ ]:
# Gerar dados sinteticos com ciclo de periodo 32 trimestres
synth = generate_trend_cycle(n=200, trend_type='linear', cycle_period=32, cycle_amp=1.0, seed=42)

y_synth = synth['observed'].values
true_cycle = synth['cycle'].values
d_synth = synth['date']

# Aplicar os tres filtros
hp_t_s, hp_c_s = hp_filter(y_synth, lamb=1600)
bk_c_s = bk_filter(y_synth, low=6, high=32, trunc=12)
cf_c_s = cf_filter(y_synth, low=6, high=32)

# Grafico 6: Comparacao com ciclo verdadeiro
K = 12
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for ax, name, est_cycle in zip(
    axes, ['HP (λ=1600)', 'BK (6-32)', 'CF (6-32)'],
    [hp_c_s, np.full(len(y_synth), np.nan), cf_c_s]
):
    ax.plot(d_synth, true_cycle, 'k--', alpha=0.5, label='Ciclo verdadeiro', linewidth=1)
    if name == 'BK (6-32)':
        ax.plot(d_synth.iloc[K:-K], bk_c_s, label=name, linewidth=1.2)
    else:
        ax.plot(d_synth, est_cycle, label=name, linewidth=1.2)
    ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax.set_title(name)
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.suptitle('Validacao: Filtros vs Ciclo Verdadeiro (Dados Sinteticos)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# Correlacoes com ciclo verdadeiro
print('Correlacao com ciclo verdadeiro:')
print(f'  HP:  {np.corrcoef(hp_c_s, true_cycle)[0,1]:.4f}')
print(f'  BK:  {np.corrcoef(bk_c_s, true_cycle[K:-K])[0,1]:.4f}')
print(f'  CF:  {np.corrcoef(cf_c_s, true_cycle)[0,1]:.4f}')

## Exercicio 1: PIB do Brasil com Diferentes Filtros

Carregue o dataset `brazil_gdp.csv` e:

1. Aplique o filtro HP com $\lambda = 1600$
2. Aplique o filtro BK com banda 6-32 trimestres
3. Aplique o filtro CF com a mesma banda
4. Compare os ciclos estimados visualmente
5. Calcule as correlacoes entre os ciclos

**Pergunta**: qual filtro indica maior amplitude ciclica? Os ciclos concordam sobre os periodos de recessao?

In [ ]:
# Exercicio 1: Seu codigo aqui
brazil_gdp = pd.read_csv(os.path.join(data_dir, 'brazil_gdp.csv'), parse_dates=['date'])

y_br = brazil_gdp['gdp_log'].values
dates_br = brazil_gdp['date']

# 1. HP filter
hp_trend_br, hp_cycle_br = hp_filter(y_br, lamb=1600)

# 2. BK filter
bk_cycle_br = bk_filter(y_br, low=6, high=32, trunc=12)

# 3. CF filter
cf_cycle_br = cf_filter(y_br, low=6, high=32)

# 4. Comparacao visual
K = 12
cycles_br = {
    'HP (λ=1600)': hp_cycle_br[K:-K],
    'BK (6-32)': bk_cycle_br,
    'CF (6-32)': cf_cycle_br[K:-K]
}

fig = plot_bandpass_comparison(
    dates_br.iloc[K:-K], cycles_br,
    title='Ciclo do PIB Real do Brasil (log) - Comparacao de Filtros'
)
plt.show()

# 5. Correlacoes
print('Correlacoes entre ciclos (Brasil):')
print(f'  HP vs BK: {np.corrcoef(hp_cycle_br[K:-K], bk_cycle_br)[0,1]:.4f}')
print(f'  HP vs CF: {np.corrcoef(hp_cycle_br[K:-K], cf_cycle_br[K:-K])[0,1]:.4f}')
print(f'  BK vs CF: {np.corrcoef(bk_cycle_br, cf_cycle_br[K:-K])[0,1]:.4f}')

print(f'\nAmplitude (std) dos ciclos:')
print(f'  HP: {hp_cycle_br.std():.4f}')
print(f'  BK: {bk_cycle_br.std():.4f}')
print(f'  CF: {cf_cycle_br.std():.4f}')

## Exercicio 2: Efeito do Parametro de Truncamento no BK

O parametro $K$ (truncation) do filtro BK controla a qualidade da aproximacao band-pass.
Valores maiores de $K$ melhoram a aproximacao, mas perdem mais observacoes.

1. Aplique o BK ao PIB dos EUA com $K = 4, 8, 12, 16$
2. Compare os ciclos resultantes
3. Qual o trade-off entre qualidade do filtro e perda de dados?

In [ ]:
# Exercicio 2: Efeito do truncamento K
fig, ax = plt.subplots(figsize=(14, 5))

for K_val in [4, 8, 12, 16]:
    bk_c = bk_filter(y, low=6, high=32, trunc=K_val)
    d = dates.iloc[K_val:-K_val]
    ax.plot(d, bk_c, label=f'K={K_val} (perde {2*K_val} obs)', linewidth=1.1)

ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.set_title('Efeito do Truncamento K no Filtro Baxter-King')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Exercicio 3: CF com e sem Drift

O filtro CF aceita um parametro `drift` que remove tendencia linear antes de filtrar.

1. Aplique CF ao PIB EUA com `drift=False` e `drift=True`
2. Compare os ciclos. Quando faz diferenca remover drift?

In [ ]:
# Exercicio 3: CF com e sem drift
cf_no_drift = cf_filter(y, low=6, high=32, drift=False)
cf_with_drift = cf_filter(y, low=6, high=32, drift=True)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(dates, cf_no_drift, label='CF (drift=False)', linewidth=1.2)
ax.plot(dates, cf_with_drift, label='CF (drift=True)', linewidth=1.2, linestyle='--')
ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.set_title('Christiano-Fitzgerald: Efeito do Parametro Drift')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Correlacao entre CF com e sem drift: {np.corrcoef(cf_no_drift, cf_with_drift)[0,1]:.4f}')
print(f'Diferenca maxima absoluta: {np.max(np.abs(cf_no_drift - cf_with_drift)):.6f}')

## 7. Conclusoes

- O filtro **HP** e simples e amplamente usado, mas tem limitacoes serias (Hamilton 2018)
- Os filtros **BK** e **CF** sao fundamentados na teoria espectral (band-pass)
- BK sacrifica observacoes nas bordas; CF adapta pesos assimetricamente
- Para ciclos de negocios, a banda 6-32 trimestres e o padrao Burns-Mitchell
- A escolha de $\lambda$ no HP e a escolha de $K$ no BK sao cruciais e devem ser justificadas

No proximo notebook, exploraremos o **filtro de Hamilton** como alternativa ao HP,
e a **decomposicao de Beveridge-Nelson** para series com raiz unitaria.